# Chapter 03: Time Methods

Time series data is everywhere — stock prices, sensor readings, web analytics, retail sales. Pandas has first-class support for datetime data, making it straightforward to parse, manipulate, and analyse temporal information.

In this notebook we will cover:

- `pd.to_datetime()` conversion
- DatetimeIndex properties (`.year`, `.month`, `.day`, `.dayofweek`, `.hour`)
- Timedelta arithmetic
- `.resample()` basics
- The `.dt` accessor for Series
- Date ranges with `pd.date_range()`
- Time-based indexing and slicing

In [ ]:
import pandas as pd
import numpy as np

## `pd.to_datetime()` — Parsing Dates

Pandas can parse a wide variety of date string formats into proper `datetime64` objects.

In [ ]:
# Parse a list of date strings
dates = pd.to_datetime(['2024-01-15', '2024-02-20', '2024-03-25'])
print(type(dates))
dates

In [ ]:
# Pandas handles many formats automatically
pd.to_datetime(['Jan 1, 2024', '2024/02/15', '03-20-2024', '20240401'])

In [ ]:
# Specify format explicitly for ambiguous or non-standard dates
pd.to_datetime(['15/01/2024', '20/02/2024'], format='%d/%m/%Y')

In [ ]:
# Convert a column in a DataFrame
df = pd.DataFrame({
    'date_str': ['2024-01-01', '2024-06-15', '2024-12-31'],
    'sales': [100, 250, 180]
})

df['date'] = pd.to_datetime(df['date_str'])
print(df.dtypes)
df

## The `.dt` Accessor

When a Series has `datetime64` dtype, the `.dt` accessor exposes datetime properties and methods — similar to how `.str` works for strings.

In [ ]:
df['date'].dt.year

In [ ]:
df['date'].dt.month

In [ ]:
df['date'].dt.day

In [ ]:
# Day of week: Monday=0, Sunday=6
df['date'].dt.dayofweek

In [ ]:
# Day name for readability
df['date'].dt.day_name()

In [ ]:
# Quarter
df['date'].dt.quarter

## `pd.date_range()` — Generating Date Sequences

In [ ]:
# Daily range
pd.date_range(start='2024-01-01', periods=7, freq='D')

In [ ]:
# Monthly range
pd.date_range(start='2024-01-01', end='2024-12-31', freq='MS')  # MS = Month Start

In [ ]:
# Business days only
pd.date_range(start='2024-01-01', periods=10, freq='B')

In [ ]:
# Hourly
pd.date_range(start='2024-01-01 08:00', periods=8, freq='h')

Common frequency aliases:

| Alias | Description |
|-------|-------------|
| `D` | Calendar day |
| `B` | Business day |
| `W` | Weekly |
| `MS` / `ME` | Month start / end |
| `QS` / `QE` | Quarter start / end |
| `YS` / `YE` | Year start / end |
| `h` | Hourly |
| `min` | Minutely |

## Timedelta — Duration Arithmetic

In [ ]:
# Subtracting two datetimes gives a Timedelta
start = pd.Timestamp('2024-01-01')
end = pd.Timestamp('2024-03-15')
delta = end - start
print(f"Duration: {delta}")
print(f"Days: {delta.days}")

In [ ]:
# Add a Timedelta to a date
start + pd.Timedelta(days=90)

In [ ]:
# Timedelta on a Series
dates = pd.Series(pd.date_range('2024-01-01', periods=5, freq='D'))
dates + pd.Timedelta(weeks=2)

## Working with the Retail Sales Dataset

Let us use a real time series dataset to demonstrate datetime indexing and resampling.

In [ ]:
sales = pd.read_csv('data/RetailSales_BeerWineLiquor.csv')
sales.head()

In [ ]:
# Convert the DATE column and set it as the index
sales['DATE'] = pd.to_datetime(sales['DATE'])
sales = sales.set_index('DATE')
sales.head()

In [ ]:
print(f"Date range: {sales.index.min()} to {sales.index.max()}")
print(f"Number of records: {len(sales)}")

## Time-based Indexing and Slicing

With a DatetimeIndex, `.loc` supports partial string indexing — you can select by year, year-month, or exact date.

In [ ]:
# Select all data for a single year
sales.loc['2000']

In [ ]:
# Select a specific month
sales.loc['2005-12']

In [ ]:
# Slice a date range
sales.loc['2010-01':'2010-06']

## `.resample()` — Time-based GroupBy

Resampling changes the frequency of a time series. Think of it as a `groupby` for datetime indices.

- **Downsampling**: higher frequency to lower (e.g., monthly to yearly) — requires aggregation
- **Upsampling**: lower frequency to higher — requires filling

In [ ]:
# Resample monthly data to yearly: total annual sales
annual = sales.resample('YE').sum()
annual.head(10)

In [ ]:
# Yearly average
sales.resample('YE').mean().head(10)

In [ ]:
# Quarterly resampling
sales.resample('QE').sum().head(8)

In [ ]:
# Multiple aggregations on resampled data
sales.resample('YE').agg(['mean', 'min', 'max']).head(10)

## DatetimeIndex Properties

In [ ]:
# Access year, month, etc. directly from the DatetimeIndex
print(f"Years in dataset: {sales.index.year.unique()[:5].tolist()} ...")
print(f"Months: {sales.index.month.unique().tolist()}")

In [ ]:
# Add year and month columns for analysis
sales['year'] = sales.index.year
sales['month'] = sales.index.month

# Average sales by month (seasonal pattern)
sales.groupby('month')['MRTSSM4453USN'].mean()

In [ ]:
# Clean up
sales = sales.drop(columns=['year', 'month'])

## Summary

In this notebook we covered time methods in pandas:

- `pd.to_datetime()` parses strings into `datetime64`; specify `format` for ambiguous dates.
- The `.dt` accessor exposes properties like `.year`, `.month`, `.day`, `.dayofweek` on datetime Series.
- `pd.date_range()` generates sequences of dates with configurable frequency.
- Timedelta represents durations; arithmetic between timestamps produces Timedeltas.
- With a DatetimeIndex, `.loc` supports partial string indexing for easy slicing by year or month.
- `.resample()` is a time-aware groupby — downsample with aggregation or upsample with filling.

Next up: **Inputs and Outputs** — reading from and writing to various file formats.